In [1]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('../data/ethusd.csv')

# Convert the 'time' column to datetime type and set it as the index
df['Datetime'] = pd.to_datetime(df['time'], unit='ms')
df.set_index('Datetime', inplace=True)

# Drop the original 'time' column as it's now redundant
df.drop('time', axis=1, inplace=True)

# Rename the columns
df.rename(columns={
    'open': 'Open',
    'close': 'Close',
    'high': 'High',
    'low': 'Low',
    'volume': 'Volume'
}, inplace=True)

# Resample to one-minute intervals and forward fill missing values
df = df.resample('1T').ffill()

# Display the first few rows of the dataframe
print(df.head())

# Display basic information about the dataframe
print(df.info())

                       Open   Close    High     Low  Volume
Datetime                                                   
2016-03-09 16:04:00  10.297  10.097  10.297  10.097    0.03
2016-03-09 16:05:00  10.297  10.097  10.297  10.097    0.03
2016-03-09 16:06:00  10.297  10.097  10.297  10.097    0.03
2016-03-09 16:07:00  10.297  10.097  10.297  10.097    0.03
2016-03-09 16:08:00  10.297  10.097  10.297  10.097    0.03
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3986964 entries, 2016-03-09 16:04:00 to 2023-10-08 09:27:00
Freq: min
Data columns (total 5 columns):
 #   Column  Dtype  
---  ------  -----  
 0   Open    float64
 1   Close   float64
 2   High    float64
 3   Low     float64
 4   Volume  float64
dtypes: float64(5)
memory usage: 182.5 MB
None


In [12]:
def mark_asia_open_zones(df, timeframe=4):
    """
    Mark the zones between Asia open and "timeframe" hours later on the dataframe.
    
    Args:
    df (pandas.DataFrame): The dataframe containing minute-by-minute price data.
    timeframe (int): The number of hours after Asia open to mark the zone.
    
    Returns:
    pandas.DataFrame: The dataframe with an additional 'Asia_Open_Zone' column.
    """
    # Ensure the index is timezone-aware in US/Eastern time
    df = df.tz_convert('US/Eastern')
    
    # Determine if each timestamp is in DST
    is_dst = df.index.map(lambda ts: ts.dst() != timedelta(0))

    # Map DST to asia_open_hour
    asia_open_hours = is_dst.map({True: 21, False: 20})

    # Compute asia_open_time and asia_close_time
    dates = df.index.normalize()
    asia_open_times = dates + pd.to_timedelta(asia_open_hours, unit='h')
    asia_close_times = asia_open_times + pd.Timedelta(hours=timeframe)

    # Create 'Asia_Open_Zone' column
    df['Asia_Open_Zone'] = (df.index >= asia_open_times) & (df.index < asia_close_times)
    
    return df

In [13]:
# Apply the function to the all_data dataframe
all_data = mark_asia_open_zones(df)

# Display the first few rows of the updated dataframe
print(all_data.head())

# Display some statistics about the Asia Open Zones
asia_open_data = all_data[all_data['Asia_Open_Zone']]
print(f"\nTotal number of minutes in Asia Open Zones: {len(asia_open_data)}")
print(f"Percentage of data in Asia Open Zones: {len(asia_open_data) / len(all_data) * 100:.2f}%")

TypeError: Cannot convert tz-naive timestamps, use tz_localize to localize

In [14]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def simulate_trading_strategy_stock(df, breakout_pct=0.2, starting_capital=10000, leverage=1, take_profit_pct=0.5, stop_loss_pct=0.5, enter_opposite_side=False):
    """
    Simulate the trading strategy with capital and leverage considerations.

    Args:
    df (pandas.DataFrame): The dataframe with minute-by-minute price data and 'Asia_Open_Zone' column.
    breakout_pct (float): Percentage of the range to consider as a breakout.
    trailing_stop_pct (float): Percentage of the range to set as the trailing stop.
    starting_capital (float): The starting capital for the trading simulation.
    leverage (float): The leverage to be used for each trade.

    Returns:
    pandas.DataFrame: A dataframe containing the details of each trade executed.
    """
    # Ensure the dataframe is sorted by index
    df = df.sort_index()

    # Create a unique ID for each Asia open session
    df['Asia_Open_Session_ID'] = (df['Asia_Open_Zone'] & (~df['Asia_Open_Zone'].shift(1).fillna(False))).cumsum()

    # Compute the high and low for each Asia open session
    asia_open_ranges = df[df['Asia_Open_Zone']].groupby('Asia_Open_Session_ID').agg({
        'High': 'max',
        'Low': 'min'
    }).rename(columns={'High': 'Asia_Open_High', 'Low': 'Asia_Open_Low'})

    # Forward-fill the session IDs and merge the Asia open ranges back into the dataframe
    df['Asia_Open_Session_ID'] = df['Asia_Open_Session_ID'].ffill()
    df = df.merge(asia_open_ranges, on='Asia_Open_Session_ID', how='left')

    # Calculate the range and breakout thresholds
    df['Asia_Open_Range'] = df['Asia_Open_High'] - df['Asia_Open_Low']
    df['Breakout_Threshold'] = breakout_pct * df['Asia_Open_Range']

    # Determine if each timestamp is after the Asia open session
    df['After_Asia_Open_Session'] = ~df['Asia_Open_Zone']

    # Initialize variables
    all_trades = []
    capital = starting_capital

    # Process each Asia open session individually
    for session_id, group in df.groupby('Asia_Open_Session_ID'):
        if capital <= 0:
            print("Capital exhausted. Stopping simulation.")
            break
        trades = process_trading_strategy_stock(group, breakout_pct, capital, leverage, take_profit_pct, stop_loss_pct, enter_opposite_side)

        for trade in trades:
            # Update capital based on the profit from the trade
            capital += trade['Profit']
            # Store capital after the trade
            trade['Capital_After_Trade'] = capital

        all_trades.extend(trades)

    # Convert the list of trades into a DataFrame
    trades_df = pd.DataFrame(all_trades)
    return trades_df, df

def process_trading_strategy_stock(
    group,
    breakout_pct,
    capital,
    leverage,
    take_profit_pct,
    stop_loss_pct,
    enter_opposite_side=False
):
    """
    Process the trading strategy for a single Asia open session with capital and leverage considerations.

    Args:
    group (pandas.DataFrame): The dataframe for a single Asia open session.
    breakout_pct (float): Percentage of the range to consider as a breakout.
    capital (float): The capital available before the trade.
    leverage (float): The leverage to be used for the trade.
    take_profit_pct (float): Percentage of the range to set as the take profit target.
    stop_loss_pct (float): Percentage of the range to set as the stop loss level.
    enter_opposite_side (bool): If True, enter at the opposite side of the range after first breakout.

    Returns:
    list: A list of dictionaries containing trade details.
    """
    # Only process data after the Asia open session
    group = group[group['After_Asia_Open_Session']]

    # Initialize variables
    in_breakout = False
    breakout_direction = None  # 'up' or 'down'
    reentered_range = False
    trade_executed = False
    trades = []

    # Get the Asia open high, low, and range
    if group.empty:
        return trades  # No data to process

    asia_open_high = group['Asia_Open_High'].iloc[0]
    asia_open_low = group['Asia_Open_Low'].iloc[0]
    asia_open_range = asia_open_high - asia_open_low
    breakout_threshold = breakout_pct * asia_open_range

    # Define breakout levels
    breakout_up_level = asia_open_high + breakout_threshold
    breakout_down_level = asia_open_low - breakout_threshold

    for idx, row in group.iterrows():
        if not in_breakout:
            # Check for initial breakout
            if row['High'] >= breakout_up_level:
                in_breakout = True
                breakout_direction = 'up'
                breakout_price = row['Close']
            elif row['Low'] <= breakout_down_level:
                in_breakout = True
                breakout_direction = 'down'
                breakout_price = row['Close']
        elif not reentered_range:
            # Check if price re-enters the range
            if (breakout_direction == 'up' and row['Low'] <= asia_open_low):
                reentered_range = True
            elif (breakout_direction == 'down' and row['High'] >= asia_open_high):
                reentered_range = True
        elif not trade_executed:
            if enter_opposite_side:
                # Enter at the opposite side of the range after re-entry
                if breakout_direction == 'up' and row['Low'] <= asia_open_low:
                    # Execute a long trade at the bottom of the range
                    trade_executed = True
                    entry_price = asia_open_low
                    trade_time = idx
                    stop_loss = entry_price - stop_loss_pct * asia_open_range
                    take_profit = entry_price + take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Long',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Leverage': leverage,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
                elif breakout_direction == 'down' and row['High'] >= asia_open_high:
                    # Execute a short trade at the top of the range
                    trade_executed = True
                    entry_price = asia_open_high
                    trade_time = idx
                    stop_loss = entry_price + stop_loss_pct * asia_open_range
                    take_profit = entry_price - take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Short',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Leverage': leverage,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
            else:
                # Original logic: Check for second breakout in the same direction
                if breakout_direction == 'up' and row['Low'] <= asia_open_low:
                    # Execute a long trade
                    trade_executed = True
                    entry_price = asia_open_low
                    trade_time = idx
                    stop_loss = entry_price - stop_loss_pct * asia_open_range
                    take_profit = asia_open_high + take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Long',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Leverage': leverage,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
                elif breakout_direction == 'down' and row['High'] >= asia_open_high:
                    # Execute a short trade
                    trade_executed = True
                    entry_price = asia_open_high
                    trade_time = idx
                    stop_loss = entry_price + stop_loss_pct * asia_open_range
                    take_profit = asia_open_low - take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Short',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Leverage': leverage,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
        elif trade_executed:
            # Manage the trade by checking stop loss and take profit
            current_trade = trades[-1]
            if current_trade['Direction'] == 'Long':
                # Check if take profit is hit
                if row['High'] >= current_trade['Take_Profit']:
                    exit_price = current_trade['Take_Profit']
                    exit_time = idx
                    profit = capital * leverage * ((exit_price - current_trade['Entry_Price']) / current_trade['Entry_Price'])
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
                # Check if stop loss is hit
                elif row['Low'] <= current_trade['Stop_Loss']:
                    exit_price = current_trade['Stop_Loss']
                    exit_time = idx
                    profit = capital * leverage * ((exit_price - current_trade['Entry_Price']) / current_trade['Entry_Price'])
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
            elif current_trade['Direction'] == 'Short':
                # Check if take profit is hit
                if row['Low'] <= current_trade['Take_Profit']:
                    exit_price = current_trade['Take_Profit']
                    exit_time = idx
                    profit = capital * leverage * ((current_trade['Entry_Price'] - exit_price) / current_trade['Entry_Price'])
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
                # Check if stop loss is hit
                elif row['High'] >= current_trade['Stop_Loss']:
                    exit_price = current_trade['Stop_Loss']
                    exit_time = idx
                    profit = capital * leverage * ((current_trade['Entry_Price'] - exit_price) / current_trade['Entry_Price'])
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed

    # If trade is still open at the end of the day, exit at the close price
    if trade_executed and np.isnan(trades[-1]['Exit_Price']):
        current_trade = trades[-1]
        exit_price = group['Close'].iloc[-1]
        exit_time = group.index[-1]
        if current_trade['Direction'] == 'Long':
            profit = capital * leverage * ((exit_price - current_trade['Entry_Price']) / current_trade['Entry_Price'])
        elif current_trade['Direction'] == 'Short':
            profit = capital * leverage * ((current_trade['Entry_Price'] - exit_price) / current_trade['Entry_Price'])
        current_trade.update({
            'Exit_Price': exit_price,
            'Exit_Time': exit_time,
            'Profit': profit
        })

    return trades


def plot_trading_chart(df, trades_df):
    """
    Plot the OHLC data on a candlestick chart using Plotly, show the opening ranges,
    and display entries and exits on the chart.

    Args:
    df (pandas.DataFrame): The original dataframe with price data and Asia open ranges.
    trades_df (pandas.DataFrame): The dataframe containing the details of each trade executed.
    """
    # Create a Plotly figure
    fig = go.Figure()

    # Add candlestick trace
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name='Price'
    ))

    # Add Asia opening ranges as horizontal boxes
    print("df:", df)
    asia_open_sessions = df[['Asia_Open_Session_ID', 'Asia_Open_High', 'Asia_Open_Low']].drop_duplicates()
    for _, session in asia_open_sessions.iterrows():
        session_id = session['Asia_Open_Session_ID']
        session_data = df[df['Asia_Open_Session_ID'] == session_id]
        # Get the time range for the session
        session_times = session_data.index
        if len(session_times) == 0:
            continue
        start_time = session_times[0]
        end_time = session_times[-1]
        asia_open_high = session['Asia_Open_High']
        asia_open_low = session['Asia_Open_Low']

        fig.add_shape(
            type="rect",
            x0=start_time,
            y0=asia_open_low,
            x1=end_time,
            y1=asia_open_high,
            line=dict(color="LightSeaGreen", width=1),
            fillcolor="LightSeaGreen",
            opacity=0.2,
            layer="below"
        )

    # Add trade entries and exits
    for idx, trade in trades_df.iterrows():
        entry_time = trade['Time']
        entry_price = trade['Entry_Price']
        exit_time = trade['Exit_Time']
        exit_price = trade['Exit_Price']
        direction = trade['Direction']

        # Entry marker
        fig.add_trace(go.Scatter(
            x=[entry_time],
            y=[entry_price],
            mode='markers',
            marker=dict(
                color='green' if direction == 'Long' else 'red',
                symbol='triangle-up' if direction == 'Long' else 'triangle-down',
                size=10
            ),
            name='Trade Entry'
        ))

        # Exit marker
        if pd.notna(exit_time):
            fig.add_trace(go.Scatter(
                x=[exit_time],
                y=[exit_price],
                mode='markers',
                marker=dict(
                    color='darkgreen' if direction == 'Long' else 'darkred',
                    symbol='x',
                    size=10
                ),
                name='Trade Exit'
            ))

            # Draw line from entry to exit
            fig.add_trace(go.Scatter(
                x=[entry_time, exit_time],
                y=[entry_price, exit_price],
                mode='lines',
                line=dict(
                    color='green' if direction == 'Long' else 'red',
                    dash='dash'
                ),
                showlegend=False
            ))

    # Update layout
    fig.update_layout(
        title='Trading Strategy Simulation',
        yaxis_title='Price',
        xaxis_title='Time',
        xaxis_rangeslider_visible=False,
        template='plotly_dark'
    )

    fig.show()

# Example usage:
if __name__ == "__main__":

    # Simulate the trading strategy with starting capital and leverage
    trades_df, df = simulate_trading_strategy_stock(all_data, breakout_pct=0.75, starting_capital=10000, leverage=20, take_profit_pct=2.5, stop_loss_pct=1, enter_opposite_side=True)

    # Calculate total profit
    total_profit = trades_df['Profit'].sum()
    print(f"Total Profit: {total_profit}")

    # Calculate win rate
    win_rate = (trades_df['Profit'] > 0).mean() * 100
    print(f"Win Rate: {win_rate:.2f}%")

    # Display the trades
    print(trades_df)

    # Plot the trading chart
    plot_trading_chart(df, trades_df)


Total Profit: 9453.138250988344
Win Rate: 57.14%
     Time Direction  Entry_Price    Stop_Loss  Take_Profit   Exit_Price  \
0     214     Short  2770.484619  2787.669922  2727.521362  2787.669922   
1    1411      Long  2514.960693  2493.765381  2567.948975  2522.752686   
2    1695     Short  2528.810303  2551.916748  2471.044189  2525.040771   
3    2277     Short  2508.069092  2538.280273  2432.541138  2432.541138   
4    4232      Long  2269.312500  2251.105225  2314.830688  2251.105225   
5    4794      Long  2336.238525  2321.922852  2372.027710  2372.027710   
6    5622     Short  2362.748779  2373.776367  2335.179810  2373.776367   
7    7155     Short  2340.162598  2364.022217  2280.513550  2364.022217   
8    8833      Long  2612.339355  2582.669434  2686.514160  2639.466309   
9   10839      Long  2607.734375  2584.739746  2665.220947  2584.739746   
10  12574     Short  2510.719971  2536.001709  2447.515625  2447.515625   
11  12905     Short  2445.471924  2465.234131  2396

In [15]:
def simulate_trading_strategy_futures(
    df,
    breakout_pct=0.2,
    starting_capital=10000,
    margin_per_contract=400,
    contract_size=10,
    take_profit_pct=0.5,
    stop_loss_pct=0.5,
    enter_opposite_side=False,
):
    """
    Simulate the trading strategy with capital and futures contract considerations.

    Args:
    df (pandas.DataFrame): The dataframe with minute-by-minute price data and 'Asia_Open_Zone' column.
    breakout_pct (float): Percentage of the range to consider as a breakout.
    starting_capital (float): The starting capital for the trading simulation.
    margin_per_contract (float): The margin required per contract.
    contract_size (int): The contract size (e.g., 10 troy ounces for MGC).
    take_profit_pct (float): Percentage of the range to set as the take profit target.
    stop_loss_pct (float): Percentage of the range to set as the stop loss level.

    Returns:
    pandas.DataFrame: A dataframe containing the details of each trade executed.
    """
    # Ensure the dataframe is sorted by index
    df = df.sort_index()

    # Create a unique ID for each Asia open session
    df['Asia_Open_Session_ID'] = (df['Asia_Open_Zone'] & (~df['Asia_Open_Zone'].shift(1).fillna(False))).cumsum()

    # Compute the high and low for each Asia open session
    asia_open_ranges = df[df['Asia_Open_Zone']].groupby('Asia_Open_Session_ID').agg({
        'High': 'max',
        'Low': 'min'
    }).rename(columns={'High': 'Asia_Open_High', 'Low': 'Asia_Open_Low'})

    # Forward-fill the session IDs and merge the Asia open ranges back into the dataframe
    df['Asia_Open_Session_ID'] = df['Asia_Open_Session_ID'].ffill()
    df = df.merge(asia_open_ranges, on='Asia_Open_Session_ID', how='left')

    # Calculate the range and breakout thresholds
    df['Asia_Open_Range'] = df['Asia_Open_High'] - df['Asia_Open_Low']
    df['Breakout_Threshold'] = breakout_pct * df['Asia_Open_Range']

    # Determine if each timestamp is after the Asia open session
    df['After_Asia_Open_Session'] = ~df['Asia_Open_Zone']

    # Initialize variables
    all_trades = []
    capital = starting_capital

    # Process each Asia open session individually
    for session_id, group in df.groupby('Asia_Open_Session_ID'):
        if capital <= 0:
            print("Capital exhausted. Stopping simulation.")
            break

        # Check if capital is sufficient for the margin
        num_contracts = capital // margin_per_contract
        if num_contracts == 0:
            print(f"Insufficient capital for margin in session {session_id}. Skipping trade.")
            continue

        trades = process_trading_strategy_futures(
            group,
            breakout_pct,
            capital,
            margin_per_contract,
            contract_size,
            take_profit_pct,
            stop_loss_pct,
            enter_opposite_side,
        )

        for trade in trades:
            # Update capital based on the profit from the trade
            capital += trade['Profit']
            # Store capital after the trade
            trade['Capital_After_Trade'] = capital

        all_trades.extend(trades)

    # Convert the list of trades into a DataFrame
    trades_df = pd.DataFrame(all_trades)
    return trades_df, df

def process_trading_strategy_futures(
    group,
    breakout_pct,
    capital,
    margin_per_contract,
    contract_size,
    take_profit_pct,
    stop_loss_pct,
    enter_opposite_side=False,
):
    """
    Process the trading strategy for a single Asia open session with futures contract considerations.

    Args:
    group (pandas.DataFrame): The dataframe for a single Asia open session.
    breakout_pct (float): Percentage of the range to consider as a breakout.
    capital (float): The capital available before the trade.
    margin_per_contract (float): The margin required per contract.
    contract_size (int): The contract size (e.g., 10 troy ounces for MGC).
    take_profit_pct (float): Percentage of the range to set as the take profit target.
    stop_loss_pct (float): Percentage of the range to set as the stop loss level.
    enter_opposite_side (bool): If True, enter at the opposite side of the range after first breakout.

    Returns:
    list: A list of dictionaries containing trade details.
    """
    # Only process data after the Asia open session
    group = group[group['After_Asia_Open_Session']]

    # Initialize variables
    in_breakout = False
    breakout_direction = None  # 'up' or 'down'
    reentered_range = False
    trade_executed = False
    trades = []

    # Get the Asia open high, low, and range
    if group.empty:
        return trades  # No data to process

    asia_open_high = group['Asia_Open_High'].iloc[0]
    asia_open_low = group['Asia_Open_Low'].iloc[0]
    asia_open_range = asia_open_high - asia_open_low
    breakout_threshold = breakout_pct * asia_open_range

    # Define breakout levels
    breakout_up_level = asia_open_high + breakout_threshold
    breakout_down_level = asia_open_low - breakout_threshold

    # Before entering a trade, check if capital is sufficient
    num_contracts = capital // margin_per_contract
    if num_contracts == 0:
        print(f"Insufficient capital for margin in session {group['Asia_Open_Session_ID'].iloc[0]}.")
        return trades  # No trade executed

    for idx, row in group.iterrows():
        if not in_breakout:
            # Check for initial breakout
            if row['High'] >= breakout_up_level:
                in_breakout = True
                breakout_direction = 'up'
                breakout_price = row['Close']
            elif row['Low'] <= breakout_down_level:
                in_breakout = True
                breakout_direction = 'down'
                breakout_price = row['Close']
        elif not reentered_range:
            # Check if price re-enters the range
            if (breakout_direction == 'up' and row['Low'] <= asia_open_low):
                reentered_range = True
            elif (breakout_direction == 'down' and row['High'] >= asia_open_high):
                reentered_range = True
        elif not trade_executed:
            if enter_opposite_side:
                # Enter at the opposite side of the range after re-entry
                if breakout_direction == 'up' and row['Low'] <= asia_open_low:
                    # Execute a long trade at the bottom of the range
                    trade_executed = True
                    entry_price = asia_open_low
                    trade_time = idx
                    stop_loss = entry_price - stop_loss_pct * asia_open_range
                    take_profit = entry_price + take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Long',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Margin_Per_Contract': margin_per_contract,
                        'Num_Contracts': num_contracts,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
                elif breakout_direction == 'down' and row['High'] >= asia_open_high:
                    # Execute a short trade at the top of the range
                    trade_executed = True
                    entry_price = asia_open_high
                    trade_time = idx
                    stop_loss = entry_price + stop_loss_pct * asia_open_range
                    take_profit = entry_price - take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Short',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Margin_Per_Contract': margin_per_contract,
                        'Num_Contracts': num_contracts,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
            else:
                # Original logic: Check for second breakout in the same direction
                if breakout_direction == 'up' and row['Low'] <= asia_open_low:
                    # Execute a long trade
                    trade_executed = True
                    entry_price = asia_open_low
                    trade_time = idx
                    stop_loss = entry_price - stop_loss_pct * asia_open_range
                    take_profit = asia_open_high + take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Long',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Margin_Per_Contract': margin_per_contract,
                        'Num_Contracts': num_contracts,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
                elif breakout_direction == 'down' and row['High'] >= asia_open_high:
                    # Execute a short trade
                    trade_executed = True
                    entry_price = asia_open_high
                    trade_time = idx
                    stop_loss = entry_price + stop_loss_pct * asia_open_range
                    take_profit = asia_open_low - take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Short',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Margin_Per_Contract': margin_per_contract,
                        'Num_Contracts': num_contracts,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
        elif trade_executed:
            # Manage the trade by checking stop loss and take profit
            current_trade = trades[-1]
            if current_trade['Direction'] == 'Long':
                # Check if take profit is hit
                if row['High'] >= current_trade['Take_Profit']:
                    exit_price = current_trade['Take_Profit']
                    exit_time = idx
                    profit = num_contracts * (exit_price - current_trade['Entry_Price']) * contract_size
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
                # Check if stop loss is hit
                elif row['Low'] <= current_trade['Stop_Loss']:
                    exit_price = current_trade['Stop_Loss']
                    exit_time = idx
                    profit = num_contracts * (exit_price - current_trade['Entry_Price']) * contract_size
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
            elif current_trade['Direction'] == 'Short':
                # Check if take profit is hit
                if row['Low'] <= current_trade['Take_Profit']:
                    exit_price = current_trade['Take_Profit']
                    exit_time = idx
                    profit = num_contracts * (current_trade['Entry_Price'] - exit_price) * contract_size
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
                # Check if stop loss is hit
                elif row['High'] >= current_trade['Stop_Loss']:
                    exit_price = current_trade['Stop_Loss']
                    exit_time = idx
                    profit = num_contracts * (current_trade['Entry_Price'] - exit_price) * contract_size
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed

    # If trade is still open at the end of the day, exit at the close price
    if trade_executed and np.isnan(trades[-1]['Exit_Price']):
        current_trade = trades[-1]
        exit_price = group['Close'].iloc[-1]
        exit_time = group.index[-1]
        if current_trade['Direction'] == 'Long':
            profit = num_contracts * (exit_price - current_trade['Entry_Price']) * contract_size
        elif current_trade['Direction'] == 'Short':
            profit = num_contracts * (current_trade['Entry_Price'] - exit_price) * contract_size
        current_trade.update({
            'Exit_Price': exit_price,
            'Exit_Time': exit_time,
            'Profit': profit
        })

    return trades


# In your main function or script
if __name__ == "__main__":
    # Simulate the trading strategy with futures contract considerations
    trades_df, df = simulate_trading_strategy_futures(
        all_data,
        breakout_pct=1,
        starting_capital=10000,
        margin_per_contract=2060.49,  # Adjust as per current margin requirements
        contract_size=10,
        take_profit_pct=2.5,
        stop_loss_pct=1,
        enter_opposite_side=True,
    )

    # Calculate total profit
    total_profit = trades_df['Profit'].sum()
    print(f"Total Profit: {total_profit}")

    # Calculate win rate
    win_rate = (trades_df['Profit'] > 0).mean() * 100
    print(f"Win Rate: {win_rate:.2f}%")

    # Display the trades
    print(trades_df)

    # Plot the trading chart
    plot_trading_chart(df, trades_df)


Total Profit: 1955.435791015625
Win Rate: 50.00%
     Time Direction  Entry_Price    Stop_Loss  Take_Profit   Exit_Price  \
0     214     Short  2770.484619  2787.669922  2727.521362  2787.669922   
1    1411      Long  2514.960693  2493.765381  2567.948975  2522.752686   
2    1695     Short  2528.810303  2551.916748  2471.044189  2525.040771   
3    2277     Short  2508.069092  2538.280273  2432.541138  2432.541138   
4    4232      Long  2269.312500  2251.105225  2314.830688  2251.105225   
5    4794      Long  2336.238525  2321.922852  2372.027710  2372.027710   
6    5622     Short  2362.748779  2373.776367  2335.179810  2373.776367   
7    7155     Short  2340.162598  2364.022217  2280.513550  2364.022217   
8   10839      Long  2607.734375  2584.739746  2665.220947  2584.739746   
9   12574     Short  2510.719971  2536.001709  2447.515625  2447.515625   
10  12905     Short  2445.471924  2465.234131  2396.066406  2435.376953   
11  14872     Short  2632.541504  2656.352539  2573